dfs

In [29]:
from experta import *

class State(Fact):
    pass

class BridgeExpertSystem(KnowledgeEngine):
    def __init__(self):
        super().__init__()
        self.persons_time = {'me': 1, 'lab': 2, 'worker': 5, 'scientist': 10}
        self.max_time = 30
        self.visited = set()
        self.tree = {}
        self.node_counter = 0

    @DefFacts()
    def _initial_state(self):
        print("\n🚀 بدأ البحث...\n")
        yield State(left=('me', 'lab', 'worker', 'scientist'), right=(), light='left', time=0, path=(), node=0, parent=None)

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path, node=MATCH.node))
    def move_me_and_lab(self, left, right, time, path, node):
        print("🔥 قاعدة move_me_and_lab تفعّلت")
        self.generate_new_state(('me', 'lab'), left, right, time, path, node)

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path, node=MATCH.node))
    def move_me_and_worker(self, left, right, time, path, node):
        print("🔥 قاعدة move_me_and_worker تفعّلت")
        self.generate_new_state(('me', 'worker'), left, right, time, path, node)

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path, node=MATCH.node))
    def move_me_and_scientist(self, left, right, time, path, node):
        print("🔥 قاعدة move_me_and_scientist تفعّلت")
        self.generate_new_state(('me', 'scientist'), left, right, time, path, node)

    @Rule(State(left=MATCH.left, right=MATCH.right, light='right', time=MATCH.time, path=MATCH.path, node=MATCH.node))
    def return_me(self, left, right, time, path, node):
        if 'me' in right:
            new_left = list(left) + ['me']
            new_right = list(right)
            new_right.remove('me')
            self.generate_state_from_lists(new_left, new_right, 'left', time + self.persons_time['me'], path, f"me returned to left in 1 min", node)

    @Rule(State(time=MATCH.time),
      TEST(lambda time: time > 30),
      salience=900)
    def remove_overtime_state(self, time):
        print(f"⛔️ تم تجاهل حالة بسبب تجاوز الوقت ({time} > 17).")
        self.halt()

    def generate_new_state(self, persons, left, right, time, path, parent):
        if all(p in left for p in persons):
            new_left = list(left)
            new_right = list(right)
            for p in persons:
                new_left.remove(p)
                new_right.append(p)
            new_time = time + max(self.persons_time[p] for p in persons)
            step = f"{' and '.join(persons)} crossed to right in {new_time - time} min"
            if new_time <= self.max_time:
                self.generate_state_from_lists(new_left, new_right, 'right', new_time, path, step, parent)

    def generate_state_from_lists(self, left_list, right_list, light, time, path, step, parent):
        print(f"Generated new state: Left={left_list}, Right={right_list}, Light={light}, Time={time}, Step={step}")

        signature = (tuple(sorted(left_list)), tuple(sorted(right_list)), light)
        if signature in self.visited:
            print("⚠️ تم تجاهل حالة مكررة.")
            return
        self.visited.add(signature)
        self.node_counter += 1
        node = self.node_counter
        self.tree[node] = {'parent': parent, 'left': tuple(left_list), 'right': tuple(right_list), 'light': light, 'time': time}
        new_path = list(path) + [step]
        self.declare(State(left=tuple(left_list), right=tuple(right_list), light=light, time=time, path=tuple(new_path), node=node, parent=parent))

    @Rule(State(left=MATCH.left, right=MATCH.right, light='right', time=MATCH.time, path=MATCH.path),
          TEST(lambda left, right: set(left) == set() and set(right) == {'me', 'lab', 'worker', 'scientist'}),
          salience=100)
    def goal_reached(self, left, right, time, path):
        print("\n✅ تم الوصول إلى الهدف في:", time, "دقيقة")
        print("\n📜 خطوات الحل:")
        for i, step in enumerate(path, 1):
            print(f" {i}. {step}")
        print("\n🌳 شجرة البحث:")
        for node_id, data in self.tree.items():
            print(f"🔸 Node {node_id} (Parent: {data['parent']}) | Left: {data['left']} | Right: {data['right']} | Light: {data['light']} | Time: {data['time']}")
        self.halt()

# --- تنفيذ النظام ---
engine = BridgeExpertSystem()
engine.reset()
engine.run()



🚀 بدأ البحث...

🔥 قاعدة move_me_and_lab تفعّلت
Generated new state: Left=['worker', 'scientist'], Right=['me', 'lab'], Light=right, Time=2, Step=me and lab crossed to right in 2 min
Generated new state: Left=['worker', 'scientist', 'me'], Right=['lab'], Light=left, Time=3, Step=me returned to left in 1 min
🔥 قاعدة move_me_and_lab تفعّلت
🔥 قاعدة move_me_and_scientist تفعّلت
Generated new state: Left=['worker'], Right=['lab', 'me', 'scientist'], Light=right, Time=13, Step=me and scientist crossed to right in 10 min
Generated new state: Left=['worker', 'me'], Right=['lab', 'scientist'], Light=left, Time=14, Step=me returned to left in 1 min
🔥 قاعدة move_me_and_lab تفعّلت
🔥 قاعدة move_me_and_scientist تفعّلت
🔥 قاعدة move_me_and_worker تفعّلت
Generated new state: Left=[], Right=['lab', 'scientist', 'me', 'worker'], Light=right, Time=19, Step=me and worker crossed to right in 5 min

✅ تم الوصول إلى الهدف في: 19 دقيقة

📜 خطوات الحل:
 1. me and lab crossed to right in 2 min
 2. me returned to

bfs

In [30]:
from experta import *

class State(Fact):
    pass

class BridgeExpertSystem(KnowledgeEngine):
    def __init__(self):
        super().__init__()
        self.persons_time = {'me': 1, 'lab': 2, 'worker': 5, 'scientist': 10}
        self.max_time = 30
        self.visited = set()
        self.tree = {}
        self.node_counter = 0

    @DefFacts()
    def _initial_state(self):
        print("\n🚀 بدأ البحث...\n")
        yield State(left=('me', 'lab', 'worker', 'scientist'), right=(), light='left', time=0, path=(), node=0, parent=None)

    # --- قواعد بحكم الزمن لتفعيل البحث بالعرض ---

    # للحركات بين 0 و 10 دقائق، salience=100 (أعلى أولوية)
    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path, node=MATCH.node),
          TEST(lambda time: 0 <= time <= 10),
          salience=100)
    def move_me_and_lab_early(self, left, right, time, path, node):
        print("🔥 قاعدة move_me_and_lab (early) تفعّلت")
        self.generate_new_state(('me', 'lab'), left, right, time, path, node)

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path, node=MATCH.node),
          TEST(lambda time: 0 <= time <= 10),
          salience=100)
    def move_me_and_worker_early(self, left, right, time, path, node):
        print("🔥 قاعدة move_me_and_worker (early) تفعّلت")
        self.generate_new_state(('me', 'worker'), left, right, time, path, node)

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path, node=MATCH.node),
          TEST(lambda time: 0 <= time <= 10),
          salience=100)
    def move_me_and_scientist_early(self, left, right, time, path, node):
        print("🔥 قاعدة move_me_and_scientist (early) تفعّلت")
        self.generate_new_state(('me', 'scientist'), left, right, time, path, node)

    # للحركات بين 11 و 20 دقائق، salience=90 (أولوية أقل)
    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path, node=MATCH.node),
          TEST(lambda time: 11 <= time <= 20),
          salience=90)
    def move_me_and_lab_mid(self, left, right, time, path, node):
        print("🔥 قاعدة move_me_and_lab (mid) تفعّلت")
        self.generate_new_state(('me', 'lab'), left, right, time, path, node)

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path, node=MATCH.node),
          TEST(lambda time: 11 <= time <= 20),
          salience=90)
    def move_me_and_worker_mid(self, left, right, time, path, node):
        print("🔥 قاعدة move_me_and_worker (mid) تفعّلت")
        self.generate_new_state(('me', 'worker'), left, right, time, path, node)

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path, node=MATCH.node),
          TEST(lambda time: 11 <= time <= 20),
          salience=90)
    def move_me_and_scientist_mid(self, left, right, time, path, node):
        print("🔥 قاعدة move_me_and_scientist (mid) تفعّلت")
        self.generate_new_state(('me', 'scientist'), left, right, time, path, node)

    # للحركات بين 21 و 30 دقائق، salience=80 (أولوية أقل)
    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path, node=MATCH.node),
          TEST(lambda time: 21 <= time <= 30),
          salience=80)
    def move_me_and_lab_late(self, left, right, time, path, node):
        print("🔥 قاعدة move_me_and_lab (late) تفعّلت")
        self.generate_new_state(('me', 'lab'), left, right, time, path, node)

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path, node=MATCH.node),
          TEST(lambda time: 21 <= time <= 30),
          salience=80)
    def move_me_and_worker_late(self, left, right, time, path, node):
        print("🔥 قاعدة move_me_and_worker (late) تفعّلت")
        self.generate_new_state(('me', 'worker'), left, right, time, path, node)

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path, node=MATCH.node),
          TEST(lambda time: 21 <= time <= 30),
          salience=80)
    def move_me_and_scientist_late(self, left, right, time, path, node):
        print("🔥 قاعدة move_me_and_scientist (late) تفعّلت")
        self.generate_new_state(('me', 'scientist'), left, right, time, path, node)

    # قاعدة رجوع me للجانب اليسار - تفعّل دائماً (بدون تقسيم زمن)
    @Rule(State(left=MATCH.left, right=MATCH.right, light='right', time=MATCH.time, path=MATCH.path, node=MATCH.node),
          salience=95)
    def return_me(self, left, right, time, path, node):
        if 'me' in right:
            new_left = list(left) + ['me']
            new_right = list(right)
            new_right.remove('me')
            self.generate_state_from_lists(new_left, new_right, 'left', time + self.persons_time['me'], path, f"me returned to left in 1 min", node)

    @Rule(State(time=MATCH.time),
          TEST(lambda time: time > 30),
          salience=900)
    def remove_overtime_state(self, time):
        print(f"⛔️ تم تجاهل حالة بسبب تجاوز الوقت ({time} > 30).")
        self.halt()

    def generate_new_state(self, persons, left, right, time, path, parent):
        if all(p in left for p in persons):
            new_left = list(left)
            new_right = list(right)
            for p in persons:
                new_left.remove(p)
                new_right.append(p)
            new_time = time + max(self.persons_time[p] for p in persons)
            step = f"{' and '.join(persons)} crossed to right in {new_time - time} min"
            if new_time <= self.max_time:
                self.generate_state_from_lists(new_left, new_right, 'right', new_time, path, step, parent)

    def generate_state_from_lists(self, left_list, right_list, light, time, path, step, parent):
        print(f"Generated new state: Left={left_list}, Right={right_list}, Light={light}, Time={time}, Step={step}")

        signature = (tuple(sorted(left_list)), tuple(sorted(right_list)), light)
        if signature in self.visited:
            print("⚠️ تم تجاهل حالة مكررة.")
            return
        self.visited.add(signature)
        self.node_counter += 1
        node = self.node_counter
        self.tree[node] = {'parent': parent, 'left': tuple(left_list), 'right': tuple(right_list), 'light': light, 'time': time}
        new_path = list(path) + [step]
        self.declare(State(left=tuple(left_list), right=tuple(right_list), light=light, time=time, path=tuple(new_path), node=node, parent=parent))

    @Rule(State(left=MATCH.left, right=MATCH.right, light='right', time=MATCH.time, path=MATCH.path),
          TEST(lambda left, right: set(left) == set() and set(right) == {'me', 'lab', 'worker', 'scientist'}),
          salience=110)
    def goal_reached(self, left, right, time, path):
        print("\n✅ تم الوصول إلى الهدف في:", time, "دقيقة")
        print("\n📜 خطوات الحل:")
        for i, step in enumerate(path, 1):
            print(f" {i}. {step}")
        print("\n🌳 شجرة البحث:")
        for node_id, data in self.tree.items():
            print(f"🔸 Node {node_id} (Parent: {data['parent']}) | Left: {data['left']} | Right: {data['right']} | Light: {data['light']} | Time: {data['time']}")
        self.halt()


# --- تنفيذ النظام ---
engine = BridgeExpertSystem()
engine.reset()
engine.run()



🚀 بدأ البحث...

🔥 قاعدة move_me_and_scientist (early) تفعّلت
Generated new state: Left=['lab', 'worker'], Right=['me', 'scientist'], Light=right, Time=10, Step=me and scientist crossed to right in 10 min
🔥 قاعدة move_me_and_lab (early) تفعّلت
Generated new state: Left=['worker', 'scientist'], Right=['me', 'lab'], Light=right, Time=2, Step=me and lab crossed to right in 2 min
🔥 قاعدة move_me_and_worker (early) تفعّلت
Generated new state: Left=['lab', 'scientist'], Right=['me', 'worker'], Light=right, Time=5, Step=me and worker crossed to right in 5 min
Generated new state: Left=['lab', 'scientist', 'me'], Right=['worker'], Light=left, Time=6, Step=me returned to left in 1 min
🔥 قاعدة move_me_and_scientist (early) تفعّلت
Generated new state: Left=['lab'], Right=['worker', 'me', 'scientist'], Light=right, Time=16, Step=me and scientist crossed to right in 10 min
🔥 قاعدة move_me_and_lab (early) تفعّلت
Generated new state: Left=['scientist'], Right=['worker', 'me', 'lab'], Light=right, Tim